# 23.7 — Multi-agent coordination & communication

Communication is valuable when it correlates actions around the same equilibrium rather than merely adding information. Save a copy to Drive to edit.

This notebook turns the Stag/Hare lesson into algorithms for thresholds, correlated public signals, noisy private messages, and larger convention games. The key metric is coordination rate or expected payoff, not isolated action accuracy.

In [ ]:
import itertools
import math

import matplotlib.pyplot as plt
import numpy as np

rng = np.random.default_rng(23)

np.set_printoptions(precision=3, suppress=True)


## The concept, built once: coordination thresholds

In the lesson's game $A=egin{bmatrix}4&0\0&3\end{bmatrix}$, the row player compares $U(	ext{Stag})=4q$ with $U(	ext{Hare})=3(1-q)$. Solving $4q=3(1-q)$ gives the exact threshold $q=3/7=0.428571$.

In [ ]:
def coordination_payoff(q, stag_reward=4.0, hare_reward=3.0):
    stag_value = stag_reward * q
    hare_value = hare_reward * (1.0 - q)
    action = "Stag" if stag_value > hare_value else "Hare"
    return stag_value, hare_value, action


def coordination_threshold(stag_reward=4.0, hare_reward=3.0):
    return hare_reward / (stag_reward + hare_reward)


threshold = coordination_threshold()
stag_value, hare_value, action = coordination_payoff(threshold)

assert round(threshold, 6) == 0.428571
assert round(stag_value, 6) == round(hare_value, 6)

print("Threshold:", threshold)
print("Values at threshold:", stag_value, hare_value)


## Public signals and noisy private messages

A public coin that both agents follow gives expected payoff $0.5\cdot4+0.5\cdot3=3.5$. Independent randomization gives $0.25\cdot4+0.25\cdot3=1.75$. With message accuracy $a$, noisy matching is $P(	ext{match})=a^2+(1-a)^2$, so $a=0.8$ gives $0.68$.

In [ ]:
def public_signal_payoff(stag_reward=4.0, hare_reward=3.0, p_stag=0.5):
    return p_stag * stag_reward + (1.0 - p_stag) * hare_reward


def independent_random_payoff(stag_reward=4.0, hare_reward=3.0, p_stag=0.5):
    return p_stag * p_stag * stag_reward + (1.0 - p_stag) * (1.0 - p_stag) * hare_reward


def signal_policy(a):
    match_probability = a * a + (1.0 - a) * (1.0 - a)
    mismatch_probability = 1.0 - match_probability
    expected_payoff = a * a * 4.0 + (1.0 - a) * (1.0 - a) * 3.0
    return match_probability, mismatch_probability, expected_payoff


public_payoff = public_signal_payoff()
independent_payoff = independent_random_payoff()
match_08, mismatch_08, noisy_payoff_08 = signal_policy(0.8)
match_05, mismatch_05, noisy_payoff_05 = signal_policy(0.5)

assert round(public_payoff, 6) == 3.5
assert round(independent_payoff, 6) == 1.75
assert round(public_payoff - independent_payoff, 6) == 1.75
assert round(match_08, 6) == 0.68
assert round(match_05, 6) == 0.5

print("Public signal payoff:", public_payoff)
print("Independent payoff:", independent_payoff)
print("Noisy match probabilities:", match_08, match_05)


## Dataset ladder: D1–D5 coordination games of rising noise and scale

D1 is the 2x2 Stag/Hare game, D2 adds a public recommendation, D3 adds noisy private messages, D4 expands to many agents with conventions, and D5 combines asymmetry with noisy communication.

In [ ]:
def multi_agent_match_rate(num_agents, accuracy, trials=2000, seed=237):
    local_rng = np.random.default_rng(seed)
    intended = local_rng.integers(0, 2, size=trials)
    correct = local_rng.random((trials, num_agents)) < accuracy
    received = np.where(correct, intended[:, None], 1 - intended[:, None])
    all_match = np.all(received == received[:, :1], axis=1)
    return float(np.mean(all_match))


def build_coordination_ladder():
    return [
        {
            "name": "D1 Stag/Hare threshold",
            "agents": 2,
            "accuracy": 1.0,
            "mode": "threshold",
            "q": threshold,
        },
        {
            "name": "D2 public signal recommendations",
            "agents": 2,
            "accuracy": 1.0,
            "mode": "public",
            "q": 0.5,
        },
        {
            "name": "D3 noisy private messages",
            "agents": 2,
            "accuracy": 0.8,
            "mode": "noisy",
            "q": 0.5,
        },
        {
            "name": "D4 four-agent convention",
            "agents": 4,
            "accuracy": 0.85,
            "mode": "many",
            "q": 0.7,
        },
        {
            "name": "D5 asymmetric noisy communication",
            "agents": 6,
            "accuracy": 0.72,
            "mode": "asymmetric",
            "q": 0.45,
        },
    ]


coordination_ladder = build_coordination_ladder()
for rung in coordination_ladder:
    print(rung["name"])
    print("  agents:", rung["agents"], "accuracy:", rung["accuracy"], "belief q:", rung["q"])


In [ ]:
coordination_results = []
for rung in coordination_ladder:
    stag_value, hare_value, best_action = coordination_payoff(rung["q"])
    if rung["mode"] == "threshold":
        coord_rate = 1.0
        payoff = max(stag_value, hare_value)
    elif rung["mode"] == "public":
        coord_rate = 1.0
        payoff = public_signal_payoff()
    elif rung["agents"] == 2:
        coord_rate, mismatch, payoff = signal_policy(rung["accuracy"])
    else:
        coord_rate = multi_agent_match_rate(rung["agents"], rung["accuracy"], seed=237 + rung["agents"])
        payoff = coord_rate * public_signal_payoff()
    coordination_results.append({
        "name": rung["name"],
        "agents": rung["agents"],
        "accuracy": rung["accuracy"],
        "coord_rate": coord_rate,
        "payoff": payoff,
        "best_action": best_action,
    })

print("rung | agents | accuracy | best response | coordination rate | expected payoff")
for result in coordination_results:
    print(result["name"], result["agents"], result["accuracy"], result["best_action"], round(result["coord_rate"], 3), round(result["payoff"], 3))


In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(18, 3.5))
payoff_matrix = np.array([
    [4.0, 0.0],
    [0.0, 3.0],
])
for col, rung in enumerate(coordination_ladder):
    ax = axes[col]
    if col == 0:
        matrix = payoff_matrix
        image = ax.imshow(matrix, cmap="YlGn", vmin=0, vmax=4)
        for i in range(2):
            for j in range(2):
                ax.text(j, i, str(matrix[i, j]), ha="center", va="center")
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Stag", "Hare"])
        ax.set_yticks([0, 1])
        ax.set_yticklabels(["Stag", "Hare"])
    else:
        bars = [coordination_results[col]["coord_rate"], 1.0 - coordination_results[col]["coord_rate"]]
        ax.bar(["match", "mismatch"], bars, color=["seagreen", "tomato"])
        ax.set_ylim(0, 1)
    ax.set_title(rung["name"].split()[0] + " signals")

fig.tight_layout()
plt.show()

agents = [result["agents"] for result in coordination_results]
payoffs = [result["payoff"] for result in coordination_results]
rates = [result["coord_rate"] for result in coordination_results]

plt.figure(figsize=(7, 4))
plt.plot(agents, payoffs, marker="o", label="expected payoff")
plt.plot(agents, rates, marker="s", label="coordination rate")
plt.xlabel("number of agents")
plt.ylabel("metric")
plt.title("Coordination vs. scale and message noise")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Pitfall on D5: ignoring message noise

The wrong correlated-strategy calculation assumes every agent receives the same public recommendation. The fix models accuracy $a$ and measures match probability; for two agents this is $a^2+(1-a)^2$, and for many agents it falls faster.

In [ ]:
d5 = coordination_ladder[-1]
wrong_rate = 1.0
wrong_payoff = public_signal_payoff()
fixed_rate = multi_agent_match_rate(d5["agents"], d5["accuracy"], trials=5000, seed=2399)
fixed_payoff = fixed_rate * public_signal_payoff()
two_agent_formula = signal_policy(d5["accuracy"])[0]

print("Wrong perfect-correlation rate:", wrong_rate)
print("Wrong perfect-correlation payoff:", wrong_payoff)
print("Two-agent match formula at same accuracy:", round(two_agent_formula, 3))
print("Fixed six-agent match rate:", round(fixed_rate, 3))
print("Fixed expected payoff:", round(fixed_payoff, 3))


## Evaluate it + Practice

- Metric: coordination rate and expected payoff; compare against a no-communication baseline with independent randomization.
- Cheap sanity check: at $a=0.5$, a two-agent binary message should match with probability $0.5$.
- Ablation: replace shared public signals with independent private coins and verify payoff drops from $3.5$ to $1.75$.
- Failure signals: reporting individual message accuracy instead of joint match rate, or claiming communication helps without changing action correlation.

Practice prompt 1: Sweep message accuracy from 0.5 to 1.0 and plot the two-agent match probability.

Practice prompt 2: Change the Stag reward from 4 to 6 and recompute the belief threshold.

Practice prompt 3: Compare two-agent and six-agent coordination at the same message accuracy.